# Scenario 2 Workbench: Code Generation with Claude Code

Welcome to the hands-on offline lab for **Scenario 2**. Read [`SCENARIO_GUIDE.md`](SCENARIO_GUIDE.md) first for the full explanation of `CLAUDE.md` scope, folder rules, Plan Mode, and the 6 practice questions.

### What You Will Observe in This Lab
This notebook runs [`simulation.py`](simulation.py) offline so you can see how `ClaudeCodeGenerationEngine`:
1. Routes a clear **single-file fix** to **Direct Execution**, loading only matching frontend path rules.
2. Routes a **multi-file architectural refactor** to **Plan Mode** with `context: fork` isolation.
3. Triggers the **Interview Pattern** to ask clarifying design questions when a prompt is ambiguous (such as unspecified auth).

**Memory Hook:** *"Share in the repo, load by path, plan big, fork the noise."*

In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_2_code_generation.simulation import *

print(' Scenario Workbench Initialized: Scenario 2: Code Generation & Architectural Refactoring')
print(' Zero live API calls, 100% offline determinism')


### Step 2: Compare Single-File Direct Execution vs. Multi-File Plan Mode
Notice how touching `frontend/ui/Button.tsx` loads only `frontend_react` rules and runs directly, whereas touching multiple backend files activates `PLAN_MODE` and isolated context.

In [ ]:
# Test Execution Mode Routing
engine = ClaudeCodeGenerationEngine()

# Test 1: Single file fix -> Direct Execution
task_single = CodeGenTask(
    task_id="t1",
    description="Fix typo in Button component",
    target_files=["frontend/ui/Button.tsx"],
    is_architectural_shift=False
)
outcome_single = engine.evaluate_task(task_single)
print("Single-file Mode:", outcome_single.mode_selected)
assert outcome_single.mode_selected == ExecutionMode.DIRECT_EXECUTION
assert "frontend_react" in outcome_single.injected_path_rules

# Test 2: Multi-file refactor -> Plan Mode
task_multi = CodeGenTask(
    task_id="t2",
    description="Refactor user models and controllers",
    target_files=["backend/models.py", "backend/controllers.py"],
    is_architectural_shift=True
)
outcome_multi = engine.evaluate_task(task_multi)
print("Multi-file Mode:", outcome_multi.mode_selected)
assert outcome_multi.mode_selected == ExecutionMode.PLAN_MODE
assert outcome_multi.forked_execution_used is True



### Step 3: Trigger the Interview Pattern on an Ambiguous Request
When the developer says *"Implement user auth service"* without choosing JWTs vs. session cookies, the engine pauses and presents concrete options before editing any code.

In [ ]:
# Test Interview Pattern on Ambiguous Tech Stack
task_amb = CodeGenTask(
    task_id="t3",
    description="Implement user auth service",
    target_files=["backend/api/auth.py"],
    is_architectural_shift=False
)
outcome_amb = engine.evaluate_task(task_amb)
print("Interview Required:", outcome_amb.interview_required)
print("Clarifying Options:", outcome_amb.clarification_options)
assert outcome_amb.interview_required is True
assert len(outcome_amb.clarification_options) >= 2



### Lab Complete
All offline checks passed. For full unit tests, run `uv run pytest scenarios/scenario_2_code_generation/test_scenario_2_simulation.py` in WSL Ubuntu-26.04.